# Experiment No. 06 - DCGAN on CelebA Dataset

Run cells from top to bottom. GPU recommended.

In [ ]:
!pip -q install tensorflow tensorflow-datasets matplotlib

## Aim
Train a Deep Convolutional GAN on CelebA and generate face images. GPU recommended.

In [ ]:
import tensorflow as tf, tensorflow_datasets as tfds, matplotlib.pyplot as plt
ds=tfds.load("celeb_a",split="train")
def prep(x):
    im=tf.image.resize(x["image"],(64,64))
    return (tf.cast(im,tf.float32)-127.5)/127.5
ds=ds.map(prep).take(3000).shuffle(3000).batch(64)

In [ ]:
G=tf.keras.Sequential([
 tf.keras.layers.Input((100,)),tf.keras.layers.Dense(8*8*256,use_bias=False),
 tf.keras.layers.BatchNormalization(),tf.keras.layers.LeakyReLU(),tf.keras.layers.Reshape((8,8,256)),
 tf.keras.layers.Conv2DTranspose(128,4,2,padding="same",use_bias=False),tf.keras.layers.BatchNormalization(),tf.keras.layers.LeakyReLU(),
 tf.keras.layers.Conv2DTranspose(64,4,2,padding="same",use_bias=False),tf.keras.layers.BatchNormalization(),tf.keras.layers.LeakyReLU(),
 tf.keras.layers.Conv2DTranspose(3,4,2,padding="same",activation="tanh")])
D=tf.keras.Sequential([
 tf.keras.layers.Input((64,64,3)),tf.keras.layers.Conv2D(64,4,2,padding="same"),tf.keras.layers.LeakyReLU(),tf.keras.layers.Dropout(.3),
 tf.keras.layers.Conv2D(128,4,2,padding="same"),tf.keras.layers.LeakyReLU(),tf.keras.layers.Dropout(.3),
 tf.keras.layers.Flatten(),tf.keras.layers.Dense(1)])
bce=tf.keras.losses.BinaryCrossentropy(from_logits=True)
go=tf.keras.optimizers.Adam(.0002,beta_1=.5); do=tf.keras.optimizers.Adam(.0002,beta_1=.5)
@tf.function
def step(real):
 n=tf.shape(real)[0]; z=tf.random.normal([n,100])
 with tf.GradientTape() as gt,tf.GradientTape() as dt:
  fake=G(z,training=True); ro=D(real,training=True); fo=D(fake,training=True)
  gl=bce(tf.ones_like(fo),fo); dl=bce(tf.ones_like(ro),ro)+bce(tf.zeros_like(fo),fo)
 gv=G.trainable_variables; dv=D.trainable_variables
 go.apply_gradients(zip(gt.gradient(gl,gv),gv)); do.apply_gradients(zip(dt.gradient(dl,dv),dv))
 return gl,dl
for e in range(3):
 for real in ds: gl,dl=step(real)
 print("Epoch",e+1,"G loss",float(gl),"D loss",float(dl))
samples=G(tf.random.normal([10,100]),training=False)
plt.figure(figsize=(10,4))
for i in range(10):
 plt.subplot(2,5,i+1); plt.imshow((samples[i]+1)/2); plt.axis("off")
plt.show()

## Result
The DCGAN trains Generator and Discriminator adversarially and generates synthetic faces. CelebA download/access may require dataset terms or setup.